# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
!pip -q install duckdb huggingface_hub pandas pyarrow

In [20]:
from huggingface_hub import login
from google.colab import userdata
import duckdb

token = userdata.get("HF_TOKEN")
login(token)

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{token}'
);
""")

print("✅ Setup completed successfully!")

✅ Setup completed successfully!


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [21]:
print("""
Baseline Rule

A content page should be recommended for review when:

1. It has high search visibility (many impressions).
2. It has low user engagement (few sessions).
3. It has poor average search position.

Reason Codes

HIGH_VISIBILITY
LOW_ENGAGEMENT
LOW_SEARCH_POSITION

Action Label

REFRESH_CONTENT
""")


Baseline Rule

A content page should be recommended for review when:

1. It has high search visibility (many impressions).
2. It has low user engagement (few sessions).
3. It has poor average search position.

Reason Codes

HIGH_VISIBILITY
LOW_ENGAGEMENT
LOW_SEARCH_POSITION

Action Label

REFRESH_CONTENT



In [22]:
query = """
SELECT
CASE
WHEN gsc_impressions < 100 THEN 'Low'
WHEN gsc_impressions < 500 THEN 'Medium'
ELSE 'High'
END AS impression_bucket,

COUNT(*) AS n,
AVG(gsc_clicks) AS avg_clicks

FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'

GROUP BY impression_bucket
ORDER BY n;
"""

signal1 = con.sql(query).df()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n,avg_clicks
0,High,101451,2.937566
1,Medium,537157,0.654399
2,Low,9202770,0.018722


Signal 1 Verdict: CONFIRMED

Pages with higher impressions generally receive more clicks.

This supports using impressions as part of the baseline rule.

In [23]:
query = """
SELECT
CASE
WHEN gsc_avg_position <= 10 THEN 'Good Position'
WHEN gsc_avg_position <= 20 THEN 'Average Position'
ELSE 'Poor Position'
END AS position_bucket,

COUNT(*) AS n,
AVG(gsc_clicks) AS avg_clicks

FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'

WHERE gsc_avg_position IS NOT NULL

GROUP BY position_bucket
ORDER BY position_bucket;
"""

signal2 = con.sql(query).df()

signal2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_clicks
0,Average Position,519223,0.178053
1,Good Position,2183484,0.298278
2,Poor Position,908354,0.085977


Signal 2 Verdict:

(To be decided after seeing the output.)

Relationship between search position and clicks was examined to determine whether search position should be part of the baseline rule.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [24]:
import os

query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
WHERE ga4_data_available IS TRUE;
"""

df = con.sql(query).df()

# -----------------------------
# Baseline Score
# -----------------------------

df["score"] = (
    (df["gsc_impressions"] >= 100).astype(int) * 3 +
    (df["ga4_sessions"] <= 5).astype(int) * 2 +
    (df["gsc_avg_position"] >= 10).fillna(False).astype(int)
)

# -----------------------------
# Reason Code
# -----------------------------

def reason(row):
    if row["score"] >= 5:
        return "HIGH_VISIBILITY_LOW_ENGAGEMENT"
    elif row["score"] >= 3:
        return "MODERATE_PRIORITY"
    else:
        return "LOW_PRIORITY"

df["reason_code"] = df.apply(reason, axis=1)

# -----------------------------
# Action Label
# -----------------------------

df["action_label"] = df["score"].apply(
    lambda x: "REFRESH_CONTENT" if x >= 3 else "MONITOR"
)

# -----------------------------
# Rank Queue
# -----------------------------

df = df.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)

output_file = "work/outputs/baseline_action_score.csv"

df.to_csv(output_file, index=False)

print("✅ CSV written successfully!")
print(output_file)

df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ CSV written successfully!
work/outputs/baseline_action_score.csv


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,score,reason_code,action_label
413939,2026-03-31,client_20259bd6705d81d4,content_9fe805f532cde713,207,1,13.275362,2,1,6,HIGH_VISIBILITY_LOW_ENGAGEMENT,REFRESH_CONTENT
413936,2026-03-31,client_20259bd6705d81d4,content_512065ec37d7971e,328,2,26.256098,3,3,6,HIGH_VISIBILITY_LOW_ENGAGEMENT,REFRESH_CONTENT
413951,2026-03-31,client_20259bd6705d81d4,content_df6b778e8979533a,369,1,28.972900,1,0,6,HIGH_VISIBILITY_LOW_ENGAGEMENT,REFRESH_CONTENT
413948,2026-03-31,client_20259bd6705d81d4,content_b133fe807200f7bf,211,1,24.274882,1,1,6,HIGH_VISIBILITY_LOW_ENGAGEMENT,REFRESH_CONTENT
413947,2026-03-31,client_20259bd6705d81d4,content_c7467cdb0d904f8b,624,1,37.216346,2,2,6,HIGH_VISIBILITY_LOW_ENGAGEMENT,REFRESH_CONTENT
413944,2026-03-31,client_20259bd6705d81d4,content_7bc93291f9a5cce1,375,2,36.442667,2,2,6,HIGH_VISIBILITY_LOW_ENGAGEMENT,REFRESH_CONTENT
151251,2026-03-17,client_fef1a8f436438636,content_0c4bfb897ad619da,489,0,29.208589,1,1,6,HIGH_VISIBILITY_LOW_ENGAGEMENT,REFRESH_CONTENT
151250,2026-03-17,client_fef1a8f436438636,content_bedfef521fdf0527,676,4,15.921598,3,2,6,HIGH_VISIBILITY_LOW_ENGAGEMENT,REFRESH_CONTENT
151247,2026-03-17,client_fef1a8f436438636,content_26cb0bed3617e477,480,3,14.329167,7,3,6,HIGH_VISIBILITY_LOW_ENGAGEMENT,REFRESH_CONTENT
151270,2026-03-17,client_fef1a8f436438636,content_613fc383aebcbeb4,126,2,16.611111,3,2,6,HIGH_VISIBILITY_LOW_ENGAGEMENT,REFRESH_CONTENT


In [25]:
print("Total pages analysed:", len(df))
print("Pages recommended for refresh:", (df["action_label"] == "REFRESH_CONTENT").sum())

print("\nScore Distribution")
print(df["score"].value_counts().sort_index())

Total pages analysed: 413966
Pages recommended for refresh: 254012

Score Distribution
score
0      6645
1     11513
2    141796
3     84733
4     18043
5     77453
6     73783
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [26]:
top20 = df.head(20).copy()

reviews = []

for _, row in top20.iterrows():

    # Confidence based on score
    if row["score"] >= 6:
        confidence = "High"
    elif row["score"] >= 4:
        confidence = "Medium"
    else:
        confidence = "Low"

    # Different review notes
    if row["gsc_clicks"] == 0:
        note = "This recommendation may be wrong if the page is new and has not yet received enough search traffic."
    elif row["ga4_sessions"] <= 2:
        note = "This recommendation may be wrong if the page serves a niche audience with naturally low engagement."
    else:
        note = "This recommendation may be wrong if the content has already been refreshed or if traffic changes are seasonal."

    reviews.append({
        "content_hash_id": row["content_hash_id"],
        "action": row["action_label"],
        "reason_code": row["reason_code"],
        "confidence": confidence,
        "what_would_make_it_wrong": note
    })

review_df = pd.DataFrame(reviews)

review_df

,content_hash_id,action,reason_code,confidence,what_would_make_it_wrong
0,content_9fe805f532cde713,REFRESH_CONTENT,HIGH_VISIBILITY_LOW_ENGAGEMENT,High,This recommendation may be wrong if the page s...
1,content_512065ec37d7971e,REFRESH_CONTENT,HIGH_VISIBILITY_LOW_ENGAGEMENT,High,This recommendation may be wrong if the conten...
2,content_df6b778e8979533a,REFRESH_CONTENT,HIGH_VISIBILITY_LOW_ENGAGEMENT,High,This recommendation may be wrong if the page s...
3,content_b133fe807200f7bf,REFRESH_CONTENT,HIGH_VISIBILITY_LOW_ENGAGEMENT,High,This recommendation may be wrong if the page s...
4,content_c7467cdb0d904f8b,REFRESH_CONTENT,HIGH_VISIBILITY_LOW_ENGAGEMENT,High,This recommendation may be wrong if the page s...
5,content_7bc93291f9a5cce1,REFRESH_CONTENT,HIGH_VISIBILITY_LOW_ENGAGEMENT,High,This recommendation may be wrong if the page s...
6,content_0c4bfb897ad619da,REFRESH_CONTENT,HIGH_VISIBILITY_LOW_ENGAGEMENT,High,This recommendation may be wrong if the page i...
7,content_bedfef521fdf0527,REFRESH_CONTENT,HIGH_VISIBILITY_LOW_ENGAGEMENT,High,This recommendation may be wrong if the page s...
8,content_26cb0bed3617e477,REFRESH_CONTENT,HIGH_VISIBILITY_LOW_ENGAGEMENT,High,This recommendation may be wrong if the conten...
9,content_613fc383aebcbeb4,REFRESH_CONTENT,HIGH_VISIBILITY_LOW_ENGAGEMENT,High,This recommendation may be wrong if the page s...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [27]:
print("""
Weak Picks

Some pages may receive a high baseline score even though they are not ideal candidates for content refresh.

Possible reasons include:
- Seasonal content may naturally experience temporary traffic changes.
- Recently updated pages may not yet show the impact of the update.
- Some niche pages naturally have low engagement despite being useful.
- Temporary search ranking fluctuations may reduce clicks without indicating content quality issues.

Leakage Check

No future-window information or label-derived columns were used in building the baseline score.

The baseline only uses historical search and engagement metrics available at the decision time:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

No product flags, future observations, or derived labels were used.

Therefore, the baseline is free from data leakage and represents an honest rule-based ranking approach.
""")


Weak Picks

Some pages may receive a high baseline score even though they are not ideal candidates for content refresh.

Possible reasons include:
- Seasonal content may naturally experience temporary traffic changes.
- Recently updated pages may not yet show the impact of the update.
- Some niche pages naturally have low engagement despite being useful.
- Temporary search ranking fluctuations may reduce clicks without indicating content quality issues.

Leakage Check

No future-window information or label-derived columns were used in building the baseline score.

The baseline only uses historical search and engagement metrics available at the decision time:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

No product flags, future observations, or derived labels were used.

Therefore, the baseline is free from data leakage and represents an honest rule-based ranking approach.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.